# insightflow-ai: агент-дайджест новин

Важкі обчислення за замовчуванням вимкнені (`RUN_*` = False): ноутбук читає готові результати з `eval/results/`.
Щоб порахувати заново, поставте відповідний прапорець у `True`.

In [ ]:
import json, os, subprocess, sys
from pathlib import Path

HERE = Path.cwd()
PROJECT = HERE if (HERE / "config").exists() else HERE.parent
sys.path.insert(0, str(PROJECT))
os.chdir(PROJECT)

import pandas as pd
import matplotlib.pyplot as plt

from config.config import SEED, CATEGORIES, ROOT
from src.classify import load_articles

MODELS = ["phi", "qwen3"]
RUN_CLASSIFY_EVAL = False
RUN_AGENT_EVAL = False
RUN_DEMO = False
RESULTS = ROOT / "eval" / "results"
print("SEED =", SEED, "| python", sys.version.split()[0])

## 1. Дані

In [ ]:
articles = load_articles()
print("Усього новин у БД:", len(articles))
display(articles["source"].value_counts().head(25).to_frame("новин"))
print("Слабкі мітки (категорія за фідом; mixed = без мітки):")
display(articles["feed_category"].fillna("mixed").value_counts().to_frame("новин"))
unclassified = int(articles["category"].isna().sum())
if unclassified:
    print(f"Без категорії: {unclassified} новин. Для агента й демо спочатку: python -m src.run_daily --no-ingest")

In [ ]:
labelset = pd.read_csv(ROOT / "eval" / "labelset.csv", dtype=str, keep_default_na=False)
print(f"Розмічено вручну: {(labelset.label != '').sum()} з {len(labelset)}")
display(labelset.label.replace("", "(не розмічено)").value_counts().to_frame("новин"))

## 2. Класифікація за категоріями: baseline проти LLM

In [ ]:
if RUN_CLASSIFY_EVAL:
    subprocess.run([sys.executable, "-m", "eval.eval_classify", "--models", *MODELS], cwd=ROOT, check=True)
    subprocess.run([sys.executable, "-m", "eval.compare"], cwd=ROOT, check=True)

summary_path = RESULTS / "summary_classify.csv"
if summary_path.exists():
    summary = pd.read_csv(summary_path)
    display(summary)
    ax = summary.set_index("method")[["macro_f1", "accuracy"]].plot.bar(rot=15, figsize=(8, 4), ylim=(0, 1))
    ax.set_title("Якість на ручній вибірці"); plt.tight_layout(); plt.show()
else:
    print("Ще немає результатів: запустіть `python -m eval.eval_classify --models phi qwen3` або поставте RUN_CLASSIFY_EVAL = True")

Різниця з baseline: парний bootstrap, 95 % довірчі інтервали. Різниця статистично встановлена, лише якщо інтервал не містить 0.

In [ ]:
cmp_path = RESULTS / "compare.csv"
if cmp_path.exists():
    display(pd.read_csv(cmp_path))
else:
    print("Немає compare.csv: запустіть `python -m eval.compare`")

In [ ]:
if summary_path.exists():
    ax = summary.set_index("method")[["f1_mixed", "f1_thematic"]].plot.bar(rot=15, figsize=(8, 4), ylim=(0, 1))
    ax.set_title("macro-F1 за типом фіда"); plt.tight_layout(); plt.show()

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
METHOD = "baseline_tfidf_logreg"
path = RESULTS / f"{METHOD}.json"
if path.exists():
    preds = json.loads(path.read_text())["predictions"]
    lab = labelset[(labelset.label != "") & labelset.id.isin(preds)]
    y_true, y_pred = lab.label.tolist(), [preds[i] for i in lab.id]
    ConfusionMatrixDisplay.from_predictions(y_true, y_pred, labels=CATEGORIES, xticks_rotation=45)
    plt.title(METHOD); plt.tight_layout(); plt.show()
else:
    print("Немає", path.name)

## 3. Агент (ReAct + інструменти) проти тієї самої моделі без інструментів

In [ ]:
if RUN_AGENT_EVAL:
    subprocess.run([sys.executable, "-m", "eval.eval_agent", "--models", *MODELS], cwd=ROOT, check=True)

p = RESULTS / "summary_agent.csv"
if p.exists():
    display(pd.read_csv(p))
else:
    print("Ще немає результатів: `python -m eval.eval_agent --models phi qwen3`")

In [ ]:
p = RESULTS / f"agent_{MODELS[0]}.json"
if p.exists():
    for row in json.loads(p.read_text())["rows"][:3]:
        print("ЗАПИТ:", row["question"])
        print("  агент :", row["agent"]["answer"][:300].replace("\n", " "), "| tool_ok:", row["agent"]["tool_ok"], "| grounded:", row["agent"]["grounded"])
        print("  без інструментів:", row["plain"]["answer"][:300].replace("\n", " "), "| grounded:", row["plain"]["grounded"])
        print()

## 4. Демонстрація

In [ ]:
if RUN_DEMO:
    from src.llm import LLM
    from src.agent import Agent
    from src.tools import Toolbox
    llm = LLM(MODELS[0])
    agent = Agent(llm, Toolbox().registry)
    result = agent.run("Дай зведення новин з економіки за сьогодні.", verbose=True)
    print("\nВІДПОВІДЬ:", result["answer"])
else:
    print("Поставте RUN_DEMO = True, щоб запустити агента наживо.")

In [ ]:
latest = ROOT / "reports" / "latest.md"
print(latest.read_text(encoding="utf-8")[:2500] if latest.exists() else "Немає reports/latest.md: запустіть `python -m src.run_daily`")

## 5. Висновки

Актуальні цифри: `eval/results/` і `docs/results_tables_ua.md`. Нижче підсумок за останніми замірами (тест: 137 новин).

* **Класифікація.** Загалом baseline і LLM статистично не розрізняються (інтервали різниць містять 0). Єдина встановлена перевага: Phi-4-mini на змішаних фідах (macro-F1 0,714 проти 0,358 у baseline). Baseline навчений на слабких мітках тематичних фідів і гірше переноситься на інший розподіл.
* **Моделі.** Qwen3-4B не перевершив ні baseline, ні Phi. Phi швидший: 2,2 с проти 3,5 с на новину.
* **Агент.** З інструментами 13 із 15 запитів у обох моделей, без інструментів 0 із 15. Phi і Qwen3 не розрізняються, але Phi відповідає швидше (76 с проти 192 с).
* **Помилки агента** пов'язані з якістю рубрики economics і з запитами за сьогодні на застарілих даних, а не з форматом ReAct.
* **Обмеження.** Невелика вибірка, одна людина розмічала, слабкі мітки для baseline, 15 шаблонних запитів для агента, англомовні фіди.